## Tutorial 3 - Gmsh Simulations

In this tutorial we will cover heating simulations for custom mesh grids created in Gmsh.\
**NOTE**: This part of the code is kinda janky so the tutorial is subpar. Just check the examples if this isn't helpful

In [9]:
# !pip install gmsh

In [14]:
from Beam import Beam
from Medium import Atom, Medium
from BoundaryConditions import BoundaryConditionsGmsh
import Simulation as sim
from fipy import Gmsh3D

Let's define our beam and materials.\
**NOTE**: Lengths and x0 are not required here since you define it in Gmsh. However, the medium needs to be named.

In [7]:
# --- Beam (fixed across all runs) ---
I_0 = (2e-8) / (1.602e-19)
E_0 = 5e6
L = 1e-6
Z = 1
beam = Beam(E_0, I_0, Z, L=L, W=L, dim=3, type='Rectangular')

name = 'Al'
atomic_fraction = 1 # percentange (out of 1)
Z_Al = 13
A_Al = 27 # Given in g/mol
Al_atom = Atom(name, atomic_fraction, Z_Al, A_Al)

SRIM_fileName = "Al//Hydrogen in Aluminum.txt" # relative path or absolute path
rho_Al = 2700 # Density in kg/m^3
C_Al = 900 # Specific heat capacity in J/(kg*K)
k_Al = 205.0 # Heat conductivity in W/(m*K)

aluminum = Medium(rho_Al, C_Al, k_Al, Al_atom, SRIM_fileName, 
                  name = 'Aluminum') 

We need to define the mesh grid first before we can do anything. 

**IMPORTANT**
+ Gmsh needs to be added as a PATH variable before you can define a Gmsh grid (search online how to do this)
+ Each physical volume in the mesh grid needs to be named the same as the corresponding Medium.
+ Each physical surface needs to be named the same as the corresponding boundary condition type.

For this tutorial, we will just define a simple Aluminum cube.

In [10]:
mesh_size_um = 0.2 # mesh cell length in micro meters
GEO_TEMPLATE = """
SetFactory("OpenCASCADE");
Mesh.Recombine3DAll = 1;
Mesh.MeshSizeMax = {mesh_size};
Mesh.MeshSizeMin = {mesh_size};
Box(1) = {{0, -1, -1, 2, 2, 2}};
Physical Volume("Aluminum", 25) = {{1}};
Physical Surface("BBR", 26) = {{6, 4, 5, 3}};
Physical Surface("Fixed", 27) = {{2}};
"""

geo_str = GEO_TEMPLATE.format(mesh_size=mesh_size_um)
gmsh_mesh = Gmsh3D(geo_str)

OSError: Gmsh version must be >= 2.0.

So the way FiPy reads in Gmsh grids is strange.\
You cannot directly scale the mesh just by multiplying it, so we need to do the following process:

In [3]:
mesh_scaling_factor = 1e-6 # to convert to micrometers
mesh = gmsh_mesh * 1e-6

# Need to recover the physical cells from the old mesh
mesh.physicalCells = {k: np.asarray(v, dtype=bool)
                      for k, v in gmsh_mesh.physicalCells.items()}
mesh.physicalFaces = {k: np.asarray(v, dtype=bool)
                      for k, v in gmsh_mesh.physicalFaces.items()}
mesh.exteriorFaces = np.asarray(gmsh_mesh.exteriorFaces, dtype=bool)

NameError: name 'gmsh_mesh' is not defined

Now we are ready!

#### **GMSH BOUNDARY CONDITIONS**

We have a separate 'GmshBoundaryConditions' object now for our Boundary Conditions. \
Functionally, it is pretty similar to the 2D/3D version, except now we are required to pass in the mesh on initialization. 

The allowed types are:
1) None
2) Fixed
3) BBR
4) Internal
5) Interface

Make sure these are the exact names of your physical surfaces in the Gmsh. 

**ASIDE** \
This is pretty poorly implemented. You can only define 1 of each boundary condition for each physical volume. So you cannot have two different temperature faces on the object. I am too lazy to implement this, but you would need to get the surface ID's from the mesh and then have the user input the surface ID's of the boundary conditions.

In [ ]:
 BC = BoundaryConditionsGmsh(mesh, T0=298.0, T_amb=298.0, eps=0.75)


There are also two new boundary conditions introducted
1) **Thermal Contact Resistance** -> denote the physical surface 'Internal' in the Gmsh script\
    This deals with thermal contact resistance between two materials. You need to pass in an R value
2) **Interface** -> denote the physical surface 'Interface' in the Gmsh \
    This is basically an unideal fixed temperature condition. You pass in a constant temperature into T_0 like it's a fixed temp boundary, but then you also pass in h which is the interfacial boundary conductance.

Check the PISH example code to see this in action, and the thesis for a better explanation.


#### **GMSH SIMULATION**


Now we can run the simulation. There's not much different from the previous 2D and 3D simulations, except now we do not need to define any lengths or mesh resolutions.

An interactive 3D viewer in MayAvi will pop up. 

In [11]:
dT = 500
dt = 0.001
dt_ramp = 2
units = 'um'

In [15]:
        ts, Ts = sim.heateq_solid_3d_gmsh(
            beam, aluminum, mesh, BC,
            10, T0=298.0,
            dt=dt, dt_ramp=dt_ramp, dT_target=dT,
            debug=False, view=True, 
        )

NameError: name 'mesh' is not defined